In [ ]:
import cv2
import numpy as np
from segment_anything import SamPredictor, sam_model_registry
from ultralytics import YOLO  # Assuming YOLOv10 is used

# Step 1: Load YOLO model and SAM model
yolo_model = YOLO("path/to/yolov10/weights.pt")
sam_checkpoint = "path/to/sam_checkpoint.pth"
model_type = "vit_h"  # SAM model type: 'vit_h', 'vit_l', 'vit_b'
device = "cuda" if torch.cuda.is_available() else "cpu"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device)
predictor = SamPredictor(sam)

# Step 2: Load the input image
image_path = "path/to/input/image.jpg"
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Step 3: Perform object detection using YOLO
results = yolo_model(image_path)

# Step 4: Extract bounding boxes labeled as agricultural fields
agri_field_bboxes = []
for result in results:
    for box in result.boxes:
        # Assume agricultural fields are labeled as class 0
        if box.cls[0] == 0:  # Adjust class index as needed
            x_min, y_min, x_max, y_max = map(int, box.xyxy[0].tolist())
            agri_field_bboxes.append([x_min, y_min, x_max, y_max])

# Step 5: Use SAM to segment each bounding box
predictor.set_image(image_rgb)
segmented_masks = []

for bbox in agri_field_bboxes:
    input_box = np.array(bbox)

    # Perform segmentation for the bounding box
    masks, scores, logits = predictor.predict(box=input_box, multimask_output=False)

    # Store the segmentation mask
    segmented_masks.append(masks[0])  # Taking the first mask

    # Optionally visualize the segmentation on the image
    image_rgb[masks[0] > 0] = [0, 255, 0]  # Highlight segmentation in green

# Step 6: Save or visualize the results
output_path = "segmented_fields.jpg"
cv2.imwrite(output_path, cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))
print(f"Segmented agricultural fields saved to {output_path}")
